# Week 27: Production pipelines: S3 → Lambda → EventBridge → DDB

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/27/](https://launchdetect.com/academy/week/27/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_Production geospatial isn't a notebook — it's a pipeline. This week is the real AWS architecture LaunchDetect runs in production: S3 ingest, Lambda compute, EventBridge schedule, DynamoDB state._


## Why this week matters

Notebook code is for exploration. Production code is for running every minute, every day, autonomously. This week is the bridge — and the architecture LaunchDetect actually uses for the live launch-detection pipeline.


## Learning objectives

By the end of this lab you will be able to:

- Wire S3 PutObject events to Lambda triggers
- Use EventBridge for scheduled and event-driven orchestration
- Persist detection records to DynamoDB
- Reason about cost and latency in serverless geospatial pipelines


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Write a Python function with the Lambda-handler signature. Simulate an S3 event. Wire the detection logic inside. Then in real deployment, package as a Lambda image and let CDK provision the event trigger.


In [ ]:
# Week 27: production pipeline scaffold — S3 event -> Lambda -> DynamoDB.
# (Local simulation; the real Lambda lives in launchdetect-web/infrastructure/lambda/.)

def lambda_handler(event, context):
    """Simulates an S3 PutObject event handler."""
    s3_key = event["Records"][0]["s3"]["object"]["key"]
    print(f"Processing {s3_key}")
    # 1. Download NetCDF from s3://noaa-goes18/<key>
    # 2. Convert Band 7 radiance -> brightness temperature (Week 14)
    # 3. Threshold-detect hotspots (>320 K)
    # 4. Apply parallax correction (Week 15)
    # 5. Write detection record to DynamoDB
    detections = [{"id": "DET-001", "t": "2026-05-11T15:30:00Z",
                   "lat": 28.49, "lon": -80.60, "tb_k": 342, "vehicle": "Falcon 9"}]
    return {"statusCode": 200, "body": {"detections": detections}}

mock_event = {"Records": [{"s3": {"object": {"key": "ABI-L1b-RadM/2024/130/15/test.nc"}}}]}
print(lambda_handler(mock_event, None))

# TODO: package this as a real Lambda function, deploy with AWS CDK, point
# its trigger at the noaa-goes18 bucket (or a copy in your account).


## What just happened — and why it works

S3 → Lambda → DynamoDB is the canonical AWS serverless geospatial pattern. Cold start is the main pitfall for low-latency endpoints; for event-driven batch (which this is), cold start is fine.


## Common gotchas

- Lambda has a 15-minute hard timeout. Detection per frame should be sub-second; if you're approaching minutes, refactor.
- DynamoDB partition key choice determines whether the pipeline scales. Use a high-cardinality key (detection ULID), not a date.
- Logging in Lambda goes to CloudWatch. Cost can be surprising at scale — log structured JSON, not verbose strings.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/27/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
